In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Techniki łagodzenia i tłumienia błędów

> **Note:** Wersja beta nowego modelu wykonania jest już dostępna. Ukierunkowany model wykonania zapewnia większą elastyczność podczas dostosowywania przepływu pracy łagodzenia błędów. Więcej informacji znajdziesz w przewodniku [Ukierunkowany model wykonania](/guides/directed-execution-model).


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Techniki łagodzenia i tłumienia błędów służą do poprawy jakości wyników przy skalowaniu do większych obciążeń roboczych. Na tej stronie znajdują się ogólne objaśnienia technik tłumienia i łagodzenia błędów dostępnych w Qiskit Runtime.

Poniższa komórka importuje prymityw Estimator i tworzy Backend, który zostanie użyty do inicjalizacji Estimatora w kolejnych komórkach kodu.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamiczne odsprzęganie
Obwody kwantowe są wykonywane na sprzęcie IBM&reg; jako sekwencje impulsów mikrofalowych, które muszą być zaplanowane i uruchomione w precyzyjnych odstępach czasu.
Niestety, niechciane interakcje między Qubitami mogą prowadzić do koherentnych błędów w bezczynnych Qubitach. Dynamiczne odsprzęganie działa poprzez wstawianie sekwencji impulsów w bezczynnych Qubitach, aby w przybliżeniu wyeliminować efekt tych błędów. Każda wstawiona sekwencja impulsów odpowiada operacji tożsamości, ale fizyczna obecność impulsów powoduje tłumienie błędów.
Istnieje wiele możliwych wyborów sekwencji impulsów, a to, która sekwencja jest lepsza w danym przypadku, pozostaje [aktywnym obszarem badań](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Pamiętaj, że dynamiczne odsprzęganie jest głównie przydatne w przypadku Circuit zawierających przerwy, w których niektóre Qubity są bezczynne bez żadnych operacji na nich. Jeśli operacje w Circuit są bardzo gęsto upakowane, tak że wszystkie Qubity są przez większość czasu zajęte, dodanie impulsów dynamicznego odsprzęgania może nie poprawić wydajności. W rzeczywistości może nawet pogorszyć wydajność z powodu niedoskonałości samych impulsów.

Poniższy diagram przedstawia dynamiczne odsprzęganie z sekwencją impulsów XX. Abstrakcyjny Circuit po lewej stronie jest odwzorowany na harmonogram impulsów mikrofalowych w prawym górnym rogu. W prawym dolnym rogu przedstawiono ten sam harmonogram, ale z sekwencją dwóch impulsów X wstawionych podczas okresu bezczynności pierwszego Qubitu.

![Ilustracja dynamicznego odsprzęgania](../docs/images/guides/error-mitigation-explanation/dd.avif)

Dynamiczne odsprzęganie można włączyć, ustawiając `enable` na `True` w [opcjach dynamicznego odsprzęgania](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Opcja `sequence_type` pozwala wybrać spośród kilku różnych sekwencji impulsów. Domyślny typ sekwencji to `"XX"`.

Poniższa komórka kodu pokazuje, jak włączyć dynamiczne odsprzęganie dla Estimatora i wybrać sekwencję dynamicznego odsprzęgania.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Splatanie Pauliego
Splatanie, znane również jako [randomized compiling](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), to szeroko stosowana technika przekształcania dowolnych kanałów szumu w kanały szumu o bardziej specyficznej strukturze.

Splatanie Pauliego to szczególny rodzaj splatania, który używa operacji Pauliego. Ma ono efekt przekształcania dowolnego kanału kwantowego w kanał Pauliego. Wykonane samodzielnie może łagodzić koherentny szum, ponieważ koherentny szum ma tendencję do narastania kwadratowego wraz z liczbą operacji, podczas gdy szum Pauliego narasta liniowo. Splatanie Pauliego jest często łączone z innymi technikami łagodzenia błędów, które działają lepiej z szumem Pauliego niż z dowolnym szumem.

Splatanie Pauliego jest realizowane poprzez otoczenie wybranego zestawu Gate'ów losowo wybranymi jednoQubitowymi Gate'ami Pauliego w taki sposób, że idealny efekt Gate'u pozostaje taki sam. W efekcie pojedynczy Circuit jest zastępowany losowym zbiorem Circuit, wszystkich o tym samym idealnym efekcie. Podczas próbkowania Circuit, próbki są pobierane z wielu losowych instancji, a nie tylko z jednej.

![Ilustracja splatania Pauliego](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Ponieważ większość błędów w obecnym sprzęcie kwantowym pochodzi z dwuQubitowych Gate'ów, technika ta jest często stosowana wyłącznie do (natywnych) dwuQubitowych Gate'ów. Poniższy diagram przedstawia kilka splatań Pauliego dla Gate'ów CNOT i ECR. Każdy Circuit w wierszu ma ten sam idealny efekt.

![Ilustracja splatań Gate'ów](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Splatanie Pauliego można włączyć, ustawiając `enable_gates` na `True` w [opcjach splatania](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Inne godne uwagi opcje to:

- `num_randomizations`: Liczba instancji Circuit do pobrania z zestawu splatanych Circuit.
- `shots_per_randomization`: Liczba ujęć do próbkowania z każdej instancji Circuit.

Poniższa komórka kodu pokazuje, jak włączyć splatanie Pauliego i ustawić te opcje dla Estimatora. Żadna z tych opcji nie musi być ustawiana jawnie.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Ekstrakcja błędów odczytu przez splatanie (TREX)
[Ekstrakcja błędów odczytu przez splatanie (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) łagodzi efekt błędów pomiaru podczas szacowania wartości oczekiwanych obserwowalnych Pauliego.
Opiera się na pojęciu splatanych pomiarów, które są realizowane poprzez losowe zastępowanie Gate'ów pomiarowych sekwencją składającą się z: (1) Gate'u Pauliego X, (2) pomiaru i (3) klasycznego odwrócenia bitu. Tak jak w standardowym splataniu Gate'ów, ta sekwencja jest równoważna zwykłemu pomiarowi przy braku szumu, co przedstawia poniższy diagram:

![Ilustracja splatania pomiarów](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

W obecności błędu odczytu, splatanie pomiarów ma efekt diagonalizacji macierzy transferu błędów odczytu, co ułatwia jej odwrócenie. Szacowanie macierzy transferu błędów odczytu wymaga wykonania dodatkowych Circuit kalibracyjnych, co wprowadza niewielki narzut.

TREX można włączyć, ustawiając `measure_mitigation` na `True` w [opcjach odporności Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) dla Estimatora. Opcje uczenia szumu pomiarowego opisano [tutaj](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Podobnie jak w przypadku splatania Gate'ów, możesz ustawić liczbę randomizacji Circuit i liczbę ujęć na randomizację.

Poniższa komórka kodu pokazuje, jak włączyć TREX i ustawić te opcje dla Estimatora. Żadna z tych opcji nie musi być ustawiana jawnie.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Ekstrapolacja do zera szumu (ZNE)
Ekstrapolacja do zera szumu (ZNE) to technika łagodzenia błędów przy szacowaniu wartości oczekiwanych obserwowalnych. Choć często poprawia wyniki, nie gwarantuje uzyskania nieobciążonego rezultatu.

ZNE składa się z dwóch etapów:

1. _Wzmocnienie szumu_: Oryginalny Circuit kwantowy jest wykonywany wielokrotnie przy różnych poziomach szumu.
2. _Ekstrapolacja_: Idealny wynik jest szacowany przez ekstrapolację wyników wartości oczekiwanych z szumem do granicy zerowego szumu.

Oba etapy — wzmocnienia szumu i ekstrapolacji — można realizować na wiele różnych sposobów. Qiskit Runtime implementuje wzmocnienie szumu przez „cyfrowe składanie Gate'ów", co oznacza, że dwuQubitowe Gate'y są zastępowane równoważnymi sekwencjami Gate'u i jego odwrotności. Na przykład zastąpienie unitarnego $U$ przez $U U^\dagger U$ dałoby współczynnik wzmocnienia szumu równy 3. Do ekstrapolacji możesz wybierać spośród kilku postaci funkcyjnych, w tym dopasowania liniowego lub wykładniczego.
Poniższy obraz przedstawia cyfrowe składanie Gate'ów po lewej stronie oraz procedurę ekstrapolacji po prawej.

![Ilustracja ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE można włączyć, ustawiając `zne_mitigation` na `True` w [opcjach odporności Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) dla Estimatora.
Opcje Qiskit Runtime dla ZNE opisano [tutaj](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Godne uwagi opcje to:

- `noise_factors`: Współczynniki szumu do użycia przy wzmocnieniu szumu.
- `extrapolator`: Postać funkcyjna do użycia przy ekstrapolacji.

Poniższa komórka kodu pokazuje, jak włączyć ZNE i ustawić te opcje dla Estimatora. Żadna z tych opcji nie musi być ustawiana jawnie.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Probabilistyczne wzmocnienie błędów (PEA)
Jednym z głównych wyzwań w ZNE jest dokładne wzmocnienie szumu wpływającego na docelowy Circuit. Składanie Gate'ów zapewnia łatwy sposób przeprowadzenia tego wzmocnienia, ale może być niedokładne i prowadzić do nieprawidłowych wyników. Szczegóły znajdziesz w artykule [„Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), a konkretnie na stronie 4 informacji uzupełniających. Probabilistyczne wzmocnienie błędów zapewnia dokładniejsze podejście do wzmocnienia błędów poprzez uczenie szumu.

PEA to bardziej zaawansowana technika, która przeprowadza wstępne eksperymenty w celu rekonstrukcji szumu, a następnie wykorzystuje te informacje do przeprowadzenia dokładnego wzmocnienia. Zaczyna od nauczenia się modelu splatanych szumów każdej warstwy splatających Gate'ów w Circuit przed ich uruchomieniem (patrz [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) dla odpowiednich opcji uczenia). Po fazie uczenia Circuit są wykonywane przy każdym współczynniku szumu, gdzie każda splatająca warstwa Circuit jest wzmacniana poprzez probabilistyczne wstrzykiwanie jednoQubitowego szumu proporcjonalnego do odpowiedniego nauczonego modelu szumu. Więcej szczegółów znajdziesz w artykule [„Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3).

PEA składa się z trzech etapów:
1. _Uczenie_: Uczony jest model splatanych szumów każdej warstwy splatających Gate'ów w Circuit.
1. _Wzmocnienie szumu_: Oryginalny Circuit kwantowy jest wykonywany wielokrotnie przy różnych współczynnikach szumu.
2. _Ekstrapolacja_: Idealny wynik jest szacowany przez ekstrapolację wyników wartości oczekiwanych z szumem do granicy zerowego szumu.

W przypadku eksperymentów w skali użytkowej PEA jest często najlepszym wyborem.

Ponieważ PEA jest techniką wzmocnienia szumu ZNE, musisz również włączyć ZNE, ustawiając `resilience.zne_mitigation = True`. Dodatkowe opcje [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) mogą być używane do ustawiania ekstrapolatorów, poziomów wzmocnienia i tym podobnych. PEA wymaga modelu szumu, który jest automatycznie generowany podczas korzystania z prymitywów.

Poniższy fragment kodu pokazuje przykład, w którym PEA jest używane do łagodzenia wyników zadania Estimatora:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistyczne anulowanie błędów (PEC)
Probabilistyczne anulowanie błędów (PEC) to technika łagodzenia błędów przy szacowaniu wartości oczekiwanych obserwowalnych. W przeciwieństwie do ZNE zwraca nieobciążone oszacowanie wartości oczekiwanej. Jednak zazwyczaj wiąże się z większym narzutem.

W PEC efekt idealnego docelowego Circuit jest wyrażany jako liniowa kombinacja zaszumionych Circuit, które są rzeczywiście implementowalne w praktyce:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Wynik idealnego Circuit można następnie odtworzyć, wykonując różne zaszumione instancje Circuit pobrane z losowego zestawu zdefiniowanego przez liniową kombinację. Jeśli współczynniki $\eta_i$ tworzą rozkład prawdopodobieństwa, mogą być bezpośrednio używane jako prawdopodobieństwa zestawu. W praktyce niektóre współczynniki są ujemne, więc tworzą rozkład quasi-prawdopodobieństwa. Można je nadal używać do definiowania losowego zestawu, ale istnieje narzut próbkowania związany z ujemnością rozkładu quasi-prawdopodobieństwa, charakteryzowany przez wielkość

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Narzut próbkowania to multiplikatywny współczynnik określający liczbę ujęć potrzebnych do oszacowania wartości oczekiwanej z daną precyzją w porównaniu z liczbą ujęć potrzebnych z idealnego Circuit. Skaluje się kwadratowo z $\gamma$, które z kolei skaluje się wykładniczo z głębokością Circuit.

PEC można włączyć, ustawiając `pec_mitigation` na `True` w [opcjach odporności Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) dla Estimatora.
Opcje Qiskit Runtime dla PEC opisano [tutaj](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Limit narzutu próbkowania można ustawić za pomocą opcji `max_overhead`. Pamiętaj, że ograniczenie narzutu próbkowania może spowodować, że precyzja wyniku przekroczy żądaną precyzję. Domyślna wartość `max_overhead` wynosi 100.

Poniższa komórka kodu pokazuje, jak włączyć PEC i ustawić opcję `max_overhead` dla Estimatora.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Kolejne kroki
- Zapoznaj się z [samouczkiem](/tutorials/combine-error-mitigation-techniques) dotyczącym łączenia opcji łagodzenia błędów z prymitywem Estimator.
- [Skonfiguruj łagodzenie błędów.](configure-error-mitigation)
- [Skonfiguruj tłumienie błędów.](configure-error-suppression)
- Poznaj inne [opcje](runtime-options-overview) prymitywów Qiskit Runtime.
- Zdecyduj, w jakim [trybie wykonania](execution-modes) uruchomić swoje zadanie.